In [9]:
# Instalar librosa
import sys
import subprocess
import pkg_resources

required = {'torchcodec'}
installed = {pkg.key for pkg in pkg_resources.working_set}
missing = required - installed

if missing:
    print(f"Instalando {missing}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print("torchcodec ya está instalado")

# Reiniciar kernel
print("Por favor, reinicia el kernel manualmente (Kernel → Restart)")

C:\Users\ymartinez\AppData\Local\Temp\ipykernel_760\1372155565.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


torchcodec ya está instalado
Por favor, reinicia el kernel manualmente (Kernel → Restart)


In [24]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torchaudio
import librosa
from torch.utils.data import DataLoader, TensorDataset

In [25]:
# ==========================================
# 1. DEFINICIÓN DEL AUTOENCODER (1024 -> 103)
# ==========================================
class AudioCompressor(nn.Module):
    def __init__(self):
        super(AudioCompressor, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 103)
        )
        self.decoder = nn.Sequential(
            nn.Linear(103, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def compress(self, x):
        return self.encoder(x)

In [26]:
# ==========================================
# 1. DEFINICIÓN DEL AUTOENCODER (1024 -> 103)
# ==========================================
class AudioCompressor101(nn.Module):
    def __init__(self):
        super(AudioCompressor101, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 101)
        )
        self.decoder = nn.Sequential(
            nn.Linear(101, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def compress(self, x):
        return self.encoder(x)

In [27]:
# ==========================================
# 2. EXTRACTOR DE CARACTERÍSTICAS (WavLM 24)
# ==========================================
class FeatureExtractor:
    def __init__(self):
        print("Cargando WavLM Large (24 capas)...")
        self.bundle = torchaudio.pipelines.HUBERT_LARGE  
        self.model = self.bundle.get_model()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def extract_1024(self, audio_path):
        #waveform, sample_rate = torchaudio.load(audio_path)
        #if sample_rate != self.bundle.sample_rate:
            #resampler = torchaudio.transforms.Resample(sample_rate, self.bundle.sample_rate)
            #waveform = resampler(waveform)
      
        waveform, sample_rate = librosa.load(audio_path,sr=self.bundle.sample_rate,mono=True)
        waveform = torch.from_numpy(waveform).unsqueeze(0).float()  
        
        waveform = waveform.to(self.device)
        if sample_rate != self.bundle.sample_rate:
         waveform = torchaudio.functional.resample(waveform, sample_rate, self.bundle.sample_rate) 
        waveform = waveform.to(self.device)
        
        with torch.inference_mode():
            # Extraemos todas las capas (hidden_states)
            features, _ = self.model.extract_features(waveform)
            # La última capa (24) es la que nos interesa
            last_layer = features[-1] 
            # Promedio temporal para tener un solo vector de 1024
            avg_features = torch.mean(last_layer, dim=1) 
        return avg_features.cpu()

In [28]:
# ==========================================
# 3. SCRIPT PRINCIPAL DE PROCESAMIENTO
# ==========================================
def run_full_pipeline(input_folder, output_folder):
    extractor = FeatureExtractor()
    compressor = AudioCompressor()
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    audio_files = [f for f in os.listdir(input_folder) if f.endswith('.wav')]
    
    # --- FASE 1: Extracción de 1024 para entrenamiento ---
    print(f"\nFase 1: Extrayendo rasgos de {len(audio_files)} archivos...")
    all_feats_1024 = []
    for f in audio_files:
        path = os.path.join(input_folder, f)
        feat = extractor.extract_1024(path)
        all_feats_1024.append(feat)
    
    data_tensor = torch.cat(all_feats_1024, dim=0)

    # --- FASE 2: Entrenamiento rápido del Autoencoder ---
    print("\nFase 2: Entrenando compresor para reducir a 103 dimensiones...")
    dataset = TensorDataset(data_tensor)
    loader = DataLoader(dataset, batch_size=16, shuffle=True)
    optimizer = torch.optim.LBFGS(compressor.parameters(), lr=0.1)
    criterion = nn.MSELoss()

    for epoch in range(30): # 30 épocas es suficiente para convergencia básica
        total_loss = 0
        for batch in loader:
            inputs = batch[0]
            optimizer.zero_grad()
            outputs = compressor(inputs)
            loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch+1) % 10 == 0:
            print(f"Época {epoch+1}, Error de reconstrucción: {total_loss/len(loader):.6f}")

    # --- FASE 3: Generación de archivos .npy finales ---
    print("\nFase 3: Guardando archivos .npy (103 dimensiones)...")
    compressor.eval()
    with torch.no_grad():
        for i, filename in enumerate(audio_files):
            # Comprimir el vector 1024 a 103
            feat_103 = compressor.compress(all_feats_1024[i].view(1, -1))
            feat_103_np = feat_103.numpy().flatten().astype(np.float64)

            # Formato de nombre: ES_xxx.wav -> EN_xxx.npy
            new_name = filename.replace("ES_", "EN_").replace(".wav", "_features.npy")
            np.save(os.path.join(output_folder, new_name), feat_103_np)
            
    print(f"\n¡Listo! Archivos guardados en: {output_folder}")

In [5]:
data_dir = "../data"
pred_dir = os.path.join(data_dir, "spanish_test")
feature_dir = feature_directory = os.path.join(data_dir, "english")

In [ ]:
run_full_pipeline(pred_dir, feature_dir)

Cargando WavLM Large (24 capas)...

Fase 1: Extrayendo rasgos de 200 archivos...

Fase 2: Entrenando compresor para reducir a 103 dimensiones...


In [38]:
# ==========================================
# 3. SCRIPT PRINCIPAL DE PROCESAMIENTO
# ==========================================
def run_full_pipeline101(input_folder, output_folder):
    extractor = FeatureExtractor()
    compressor = AudioCompressor101()
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    audio_files = [f for f in os.listdir(input_folder) if f.endswith('.wav')]
    
    # --- FASE 1: Extracción de 1024 para entrenamiento ---
    print(f"\nFase 1: Extrayendo rasgos de {len(audio_files)} archivos...")
    all_feats_1024 = []
    for f in audio_files:
        path = os.path.join(input_folder, f)
        feat = extractor.extract_1024(path)
        all_feats_1024.append(feat)
    
    data_tensor = torch.cat(all_feats_1024, dim=0)

    # --- FASE 2: Entrenamiento rápido del Autoencoder ---
    print("\nFase 2: Entrenando compresor para reducir a 101 dimensiones...")
    dataset = TensorDataset(data_tensor)
    loader = DataLoader(dataset, batch_size=16, shuffle=True)
    optimizer = torch.optim.RAdam(compressor.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    for epoch in range(30): # 30 épocas es suficiente para convergencia básica
        total_loss = 0
        for batch in loader:
            inputs = batch[0]
            optimizer.zero_grad()
            outputs = compressor(inputs)
            loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch+1) % 10 == 0:
            print(f"Época {epoch+1}, Error de reconstrucción: {total_loss/len(loader):.6f}")

    # --- FASE 3: Generación de archivos .npy finales ---
    print("\nFase 3: Guardando archivos .npy (101 dimensiones)...")
    compressor.eval()
    with torch.no_grad():
        for i, filename in enumerate(audio_files):
            # Comprimir el vector 1024 a 103
            feat_103 = compressor.compress(all_feats_1024[i].view(1, -1))
            feat_103_np = feat_103.numpy().flatten().astype(np.float64)

            # Formato de nombre: ES_xxx.wav -> EN_xxx.npy
            new_name = filename.replace("EN_", "ES_").replace(".wav", "_features.npy")
            np.save(os.path.join(output_folder, new_name), feat_103_np)
            
    print(f"\n¡Listo! Archivos guardados en: {output_folder}")

In [39]:
data_dir = "../data"
pred_dir = os.path.join(data_dir, "english_test")
feature_dir = feature_directory = os.path.join(data_dir, "spanish")

In [40]:
run_full_pipeline101(pred_dir, feature_dir)

Cargando WavLM Large (24 capas)...

Fase 1: Extrayendo rasgos de 240 archivos...

Fase 2: Entrenando compresor para reducir a 101 dimensiones...
Época 10, Error de reconstrucción: 93.658350
Época 20, Error de reconstrucción: 66.752041
Época 30, Error de reconstrucción: 50.083907

Fase 3: Guardando archivos .npy (101 dimensiones)...

¡Listo! Archivos guardados en: ../data\spanish
